In [1]:
import mysql.connector
import sqlite3
from mysql.connector import Error

In [2]:
import sqlite3
import pandas as pd

def execute_sql_query(db_file, query):
    """
    Выполняет SQL-запрос к локальной базе данных и возвращает результат в виде DataFrame.

    :param db_file: путь к файлу локальной базы данных SQLite
    :param query: SQL-запрос для выполнения
    :return: результат запроса в виде pandas DataFrame
    """
    # Подключаемся к базе данных
    conn = sqlite3.connect(db_file)
    
    try:
        # Выполняем запрос и возвращаем результат в виде DataFrame
        result_df = pd.read_sql_query(query, conn)
        return result_df
    except Exception as e:
        print(f"Произошла ошибка при выполнении запроса: {e}")
        return None
    finally:
        # Закрываем соединение
        conn.close()

# Пример использования:
# db_file = 'local_credit_database.db'
# query = 'SELECT * FROM member LIMIT 10;'
# result = execute_sql_query(db_file, query)
# print(result)

In [6]:
import pymysql
import sqlite3
import os

# Параметры подключения к удаленной базе
remote_config = {
    'host': 'relational.fel.cvut.cz',
    'port': 3306,
    'user': 'guest',
    'password': 'ctu-relational',
    'database': 'ErgastF1',
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor
}

# Шаг 1: Подключение к удаленной базе и получение данных
print("Подключение к удаленной базе...")
remote_conn = pymysql.connect(**remote_config)
remote_cursor = remote_conn.cursor()

# Шаг 2: Находим необходимые данные для выполнения запроса
print("Поиск данных для запроса...")

# Сначала находим Circuit de Monaco
remote_cursor.execute("""
    SELECT circuitId FROM circuits 
    WHERE name = 'Circuit de Monaco'
    LIMIT 1
""")
monaco_circuit = remote_cursor.fetchone()
if not monaco_circuit:
    raise Exception("Circuit de Monaco не найден в базе данных")
monaco_circuit_id = monaco_circuit['circuitId']
print(f"Найден Circuit de Monaco с ID: {monaco_circuit_id}")

# Находим доступные годы для гонок на этом треке
remote_cursor.execute("""
    SELECT DISTINCT year FROM races 
    WHERE circuitId = %s 
    ORDER BY year DESC
    LIMIT 10
""", (monaco_circuit_id,))
available_years = [row['year'] for row in remote_cursor.fetchall()]
print(f"Доступные годы для Circuit de Monaco: {available_years}")

# Находим последний год, где есть гонка и известен победитель
selected_year = None
selected_race_id = None
winner_driver_id = None

for year in available_years:
    # Находим гонку в этом году
    remote_cursor.execute("""
        SELECT raceId FROM races 
        WHERE circuitId = %s AND year = %s
        LIMIT 1
    """, (monaco_circuit_id, year))
    race = remote_cursor.fetchone()
    
    if not race:
        continue
        
    race_id = race['raceId']
    
    # Проверяем, есть ли результаты с позицией 1 для этой гонки
    remote_cursor.execute("""
        SELECT driverId FROM results 
        WHERE raceId = %s AND position = 1
        LIMIT 1
    """, (race_id,))
    winner = remote_cursor.fetchone()
    
    if winner:
        selected_year = year
        selected_race_id = race_id
        winner_driver_id = winner['driverId']
        print(f"✅ Найдена подходящая гонка: Circuit de Monaco, {year} год")
        print(f"ID гонки: {race_id}, ID победителя: {winner_driver_id}")
        break

if not selected_year:
    raise Exception("Не найдено подходящих гонок с известным победителем на Circuit de Monaco")

# Получаем данные победителя
remote_cursor.execute("""
    SELECT * FROM drivers 
    WHERE driverId = %s
    LIMIT 1
""", (winner_driver_id,))
winner_driver = remote_cursor.fetchone()
print(f"Найден водитель: {winner_driver['forename']} {winner_driver['surname']}")

# Собираем данные для всех необходимых таблиц
data = {}

# Данные из circuits - сначала Circuit de Monaco, затем остальные до 40 строк
remote_cursor.execute("""
    (SELECT * FROM circuits WHERE circuitId = %s)
    UNION ALL
    (SELECT * FROM circuits WHERE circuitId != %s LIMIT 39)
    LIMIT 40
""", (monaco_circuit_id, monaco_circuit_id))
data['circuits'] = remote_cursor.fetchall()
print(f"Получено {len(data['circuits'])} строк из circuits")

# Данные из races - сначала выбранная гонка Monaco, затем остальные до 40 строк
remote_cursor.execute("""
    (SELECT * FROM races WHERE circuitId = %s AND year = %s)
    UNION ALL
    (SELECT * FROM races WHERE NOT (circuitId = %s AND year = %s) LIMIT 39)
    LIMIT 40
""", (monaco_circuit_id, selected_year, monaco_circuit_id, selected_year))
data['races'] = remote_cursor.fetchall()
print(f"Получено {len(data['races'])} строк из races")

# Данные из results - сначала результат победителя выбранной гонки, затем остальные до 40 строк
remote_cursor.execute("""
    (SELECT * FROM results WHERE raceId = %s AND position = 1)
    UNION ALL
    (SELECT * FROM results WHERE NOT (raceId = %s AND position = 1) LIMIT 39)
    LIMIT 40
""", (selected_race_id, selected_race_id))
data['results'] = remote_cursor.fetchall()
print(f"Получено {len(data['results'])} строк из results")

# Данные из drivers - сначала победитель, затем остальные до 40 строк
remote_cursor.execute("""
    (SELECT * FROM drivers WHERE driverId = %s)
    UNION ALL
    (SELECT * FROM drivers WHERE driverId != %s LIMIT 39)
    LIMIT 40
""", (winner_driver_id, winner_driver_id))
data['drivers'] = remote_cursor.fetchall()
print(f"Получено {len(data['drivers'])} строк из drivers")

# Закрываем соединение с удаленной базой
remote_conn.close()

# Шаг 3: Создание локальной SQLite базы
print("\nСоздание локальной базы данных...")
if os.path.exists('ergast_f1_local.db'):
    os.remove('ergast_f1_local.db')

local_conn = sqlite3.connect('ergast_f1_local.db')
local_cursor = local_conn.cursor()

# Создаем таблицы с такой же структурой
local_cursor.execute('''
CREATE TABLE circuits (
    circuitId INTEGER PRIMARY KEY,
    circuitRef TEXT,
    name TEXT,
    location TEXT,
    country TEXT,
    lat REAL,
    lng REAL,
    alt INTEGER,
    url TEXT
)
''')

local_cursor.execute('''
CREATE TABLE races (
    raceId INTEGER PRIMARY KEY,
    year INTEGER,
    round INTEGER,
    circuitId INTEGER,
    name TEXT,
    date TEXT,
    time TEXT,
    url TEXT,
    fp1_date TEXT,
    fp1_time TEXT,
    fp2_date TEXT,
    fp2_time TEXT,
    fp3_date TEXT,
    fp3_time TEXT,
    quali_date TEXT,
    quali_time TEXT,
    sprint_date TEXT,
    sprint_time TEXT
)
''')

local_cursor.execute('''
CREATE TABLE results (
    resultId INTEGER PRIMARY KEY,
    raceId INTEGER,
    driverId INTEGER,
    constructorId INTEGER,
    number INTEGER,
    grid INTEGER,
    position INTEGER,
    positionText TEXT,
    positionOrder INTEGER,
    points REAL,
    laps INTEGER,
    time TEXT,
    milliseconds INTEGER,
    fastestLap INTEGER,
    rank INTEGER,
    fastestLapTime TEXT,
    fastestLapSpeed REAL,
    statusId INTEGER
)
''')

local_cursor.execute('''
CREATE TABLE drivers (
    driverId INTEGER PRIMARY KEY,
    driverRef TEXT,
    number INTEGER,
    code TEXT,
    forename TEXT,
    surname TEXT,
    dob TEXT,
    nationality TEXT,
    url TEXT
)
''')

local_conn.commit()

# Шаг 4: Вставка данных в локальную базу
def insert_data(table_name, rows):
    """Вставка данных в SQLite таблицу"""
    if not rows:
        return
    
    columns = list(rows[0].keys())
    columns_str = ', '.join(columns)
    placeholders = ', '.join(['?' for _ in columns])
    sql = f'INSERT INTO {table_name} ({columns_str}) VALUES ({placeholders})'
    
    data_to_insert = []
    for row in rows:
        row_values = []
        for col in columns:
            value = row[col]
            # Обрабатываем специальные случаи для NULL значений
            if value is None or value == 'None' or value == '':
                row_values.append(None)
            elif isinstance(value, (int, float)):
                row_values.append(value)
            else:
                row_values.append(str(value))
        data_to_insert.append(tuple(row_values))
    
    try:
        local_cursor.executemany(sql, data_to_insert)
        local_conn.commit()
        print(f"Вставлено {len(data_to_insert)} строк в таблицу {table_name}")
    except sqlite3.Error as e:
        print(f"Ошибка при вставке в {table_name}: {e}")
        print(f"SQL запрос: {sql}")
        # Выводим структуру первой строки для отладки
        if data_to_insert:
            print("Структура первой строки:")
            for i, col in enumerate(columns):
                print(f"  {col}: {data_to_insert[0][i]} (тип: {type(data_to_insert[0][i])})")

print("\nВставка данных в локальную базу:")
for table_name in ['circuits', 'races', 'results', 'drivers']:
    if table_name in data:
        insert_data(table_name, data[table_name])

# Шаг 5: Проверка работоспособности запроса
print("\nПроверка исходного запроса на локальной базе...")
query = f"""
SELECT d.forename, d.surname
FROM results r
JOIN races ra ON r.raceId = ra.raceId
JOIN circuits c ON ra.circuitId = c.circuitId
JOIN drivers d ON r.driverId = d.driverId
WHERE c.name = 'Circuit de Monaco'
  AND ra.year = {selected_year}
  AND r.position = 1;
"""

local_cursor.execute(query)
result = local_cursor.fetchone()
print(f"Результат запроса: {result}")

if result:
    print(f"✅ Успешно! Запрос вернул результат: {result[0]} {result[1]}")
    print(f"✅ Локальная база создана успешно в файле 'ergast_f1_local.db'")
    print(f"ℹ️  Использован год: {selected_year} (вместо 2021)")
else:
    print("❌ Ошибка: запрос вернул пустой результат")

# Закрываем соединение с локальной базой
local_conn.close()

print("\nСтатистика по таблицам в локальной базе:")
for table_name in ['circuits', 'races', 'results', 'drivers']:
    conn = sqlite3.connect('ergast_f1_local.db')
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    count = cursor.fetchone()[0]
    print(f"{table_name}: {count} строк (макс. 40)")
    conn.close()

Подключение к удаленной базе...
Поиск данных для запроса...
Найден Circuit de Monaco с ID: 6
Доступные годы для Circuit de Monaco: [2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008]
✅ Найдена подходящая гонка: Circuit de Monaco, 2017 год
ID гонки: 974, ID победителя: 20
Найден водитель: Sebastian Vettel
Получено 40 строк из circuits
Получено 40 строк из races
Получено 40 строк из results
Получено 40 строк из drivers

Создание локальной базы данных...

Вставка данных в локальную базу:
Вставлено 40 строк в таблицу circuits
Вставлено 40 строк в таблицу races
Вставлено 40 строк в таблицу results
Вставлено 40 строк в таблицу drivers

Проверка исходного запроса на локальной базе...
Результат запроса: ('Sebastian', 'Vettel')
✅ Успешно! Запрос вернул результат: Sebastian Vettel
✅ Локальная база создана успешно в файле 'ergast_f1_local.db'
ℹ️  Использован год: 2017 (вместо 2021)

Статистика по таблицам в локальной базе:
circuits: 40 строк (макс. 40)
races: 40 строк (макс. 40)
results:

In [4]:
import sqlite3
import pandas as pd
from io import StringIO
import csv

def get_database_info_and_csv():
    # Подключаемся к локальной базе данных
    conn = sqlite3.connect('local_hockey_database.db')
    cursor = conn.cursor()

    # Получаем список таблиц
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    output = StringIO()
    writer = csv.writer(output)

    for table_name in tables:
        table_name = table_name[0]
        print(f"Обработка таблицы: {table_name}")

        # Получаем структуру таблицы (CREATE TABLE)
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_table_sql = cursor.fetchone()[0]
        output.write(f"\n--- Структура таблицы {table_name} ---\n")
        output.write(f"{create_table_sql}\n")

        # Получаем и выводим все данные из таблицы
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", conn)
        output.write(f"\n--- Данные таблицы {table_name} ---\n")
        
        # Записываем заголовки
        writer.writerow(df.columns.tolist())
        # Записываем данные
        for row in df.itertuples(index=False, name=None):
            writer.writerow(row)
        output.write("\n")  # Добавляем пустую строку между таблицами

    conn.close()
    
    return output.getvalue()

# Вызов функции и вывод результата
result = get_database_info_and_csv()
print(result)

Обработка таблицы: AwardsCoaches
Обработка таблицы: AwardsMisc
Обработка таблицы: AwardsPlayers
Обработка таблицы: Coaches
Обработка таблицы: CombinedShutouts
Обработка таблицы: Goalies
Обработка таблицы: GoaliesSC
Обработка таблицы: GoaliesShootout
Обработка таблицы: HOF
Обработка таблицы: Master
Обработка таблицы: Scoring
Обработка таблицы: ScoringSC
Обработка таблицы: ScoringShootout
Обработка таблицы: ScoringSup
Обработка таблицы: SeriesPost
Обработка таблицы: TeamSplits
Обработка таблицы: TeamVsTeam
Обработка таблицы: Teams
Обработка таблицы: TeamsHalf
Обработка таблицы: TeamsPost
Обработка таблицы: TeamsSC
Обработка таблицы: abbrev

--- Структура таблицы AwardsCoaches ---
CREATE TABLE AwardsCoaches (
                coachID TEXT,
                award TEXT,
                year INTEGER,
                lgID TEXT,
                note TEXT,
                FOREIGN KEY (coachID) REFERENCES Coaches (coachID) ON DELETE CASCADE ON UPDATE CASCADE
            )

--- Данные таблицы Award

In [18]:
query = """
SELECT d.forename, d.surname
FROM results r
JOIN races ra ON r.raceId = ra.raceId
JOIN circuits c ON ra.circuitId = c.circuitId
JOIN drivers d ON r.driverId = d.driverId
WHERE c.name = 'Circuit de Monaco'
  AND r.position = 1;
"""

In [19]:
db_file = 'ergast_f1_local.db'
result = execute_sql_query(db_file, query)
print(result)

    forename surname
0  Sebastian  Vettel
